# Train the Indian document extractor on a free GPU (Colab or Kaggle)

**Before you start**
- **Colab:** Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom.
- **Kaggle:** Settings -> Accelerator -> **GPU**, and turn **Internet ON** (needed to download the model). Upload `indian-extraction.zip` as a Dataset and add it to this notebook.

Run the cells in order. Cell 5 is a quick smoke test; cell 6 is the real training.

In [ ]:
!nvidia-smi

Sat Nov 14 16:26:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Get the project files

In [ ]:
import os, glob, shutil, zipfile
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
PROJ = f"{WORK}/project"
shutil.rmtree(PROJ, ignore_errors=True)

hits = glob.glob("/kaggle/input/**/generate_data.py", recursive=True)   # Kaggle dataset
if hits:
    shutil.copytree(os.path.dirname(hits[0]), PROJ)
else:                                                                      # Colab upload
    from google.colab import files
    up = files.upload()                      # choose indian-extraction.zip
    zipfile.ZipFile(list(up)[0]).extractall(f"{WORK}/unzipped")
    src = os.path.dirname(glob.glob(f"{WORK}/unzipped/**/generate_data.py", recursive=True)[0])
    shutil.copytree(src, PROJ)

os.chdir(PROJ)
print("Project at", PROJ, os.listdir("."))

Saving indian-extraction (1).zip to indian-extraction (1).zip
Project at /content/project ['generate_data.py', 'extract.py', 'evaluate.py', 'noise.py', 'metrics.py', 'app.py', 'train_on_colab_or_kaggle.ipynb', 'schemas.py', 'requirements.txt', 'train_lora.py', 'README.md', 'modeling.py']


## 2. Install libraries (PyTorch is already installed here)

In [ ]:
!pip install -q faker peft trl datasets accelerate

## 3. Generate the synthetic data

In [ ]:
!python generate_data.py --n_train 3000

train=3000 val=200 test_in=200 test_heldout=200 -> data/

--- sample document ---
Aapke
A/c XX9674 se Rs.282.00 debit hue 02-Feb-24 ko. RELIANCE SMART. UPI ref 655050052120.
--- gold ---
{"amount_inr": 282.0, "direction": "debited", "account_last4": "9674", "counterparty": "RELIANCE SMART", "date": "2024-02-02", "upi_ref": "655050052120", "balance_inr": null}


## 4. BEFORE numbers: the untuned model

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # check the license before publishing; use Qwen/Qwen2.5-0.5B-Instruct for a faster run
!python evaluate.py --test data/test_heldout.jsonl --backend hf --model $MODEL --out results/base_heldout.json
!python evaluate.py --test data/test_in_schema.jsonl --backend hf --model $MODEL --out results/base_in_schema.json

Device: cuda | dtype: torch.float16
config.json: 100% 660/660 [00:00<00:00, 3.15MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 94.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 122MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 143MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 50.3M/3.09G [00:01<00:49, 61.9MB/s, 2.76MB/s  ]
model.safetensors: downloading bytes:   6% 170M/3.09G [00:02<00:18, 154MB/s, 12.9MB/s  ]
model.safetensors: downloading bytes:   7% 220M/3.09G [00:02<00:18, 159MB/s, 17.1MB/s  ]
model.safetensors: downloading bytes:   8% 247M/3.09G [00:02<00:16, 177MB/s, 19.8MB/s  ]
model.safetensors: downloading bytes:   9% 269M/3.09G [00:02<00:15, 185MB/s, 22.1MB/s  ]
model.safetensors: reconstructing file:   5% 145M/3.09G [00:02<00:19, 153MB/s, 2.09MB/s  ]  
model.safeten

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 5. Smoke test (a few minutes). If this fails, stop and read the error.

In [ ]:
!python train_lora.py --model $MODEL --max_samples 100 --epochs 1 --batch 2 --grad_accum 8 --out outputs/smoke

Training on device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:02<00:00, 141.33it/s]
Adding EOS to train dataset: 100% 100/100 [00:00<00:00, 5001.14 examples/s]
Tokenizing train dataset: 100% 100/100 [00:00<00:00, 749.51 examples/s]
Building labels for train dataset: 100% 100/100 [00:00<00:00, 2328.10 examples/s]
Truncating train dataset: 100% 100/100 [00:00<00:00, 2960.57 examples/s]
Dropping fully masked examples from train dataset: 100% 100/100 [00:00<00:00, 4229.28 examples/s]
Adding EOS to eval dataset: 100% 20/20 [00:00<00:00, 5444.32 examples/s]
Tokenizing eval dataset: 100% 20/20 [00:00<00:00, 447.78 examples/s]
Building labels for eval dataset: 100% 20/20 [00:00<00:00, 916.10 examples/s]
Truncating eval dataset: 100% 20/20 [00:00<00:00, 1128.21 examples/s]
Dropping fully masked examples from eval dataset: 100% 20/20 [00:00<00:00, 1571.43 examples/s]
100% 7/7 [00:38<00:00,  4.14s/it]
  0% 0/3 [00:00<?, ?it/s]
 67% 2/3

## 6. Real training. Watch the loss go down. Time depends on the GPU, so check the progress bar.

In [ ]:
!python train_lora.py --model $MODEL --epochs 2 --batch 4 --grad_accum 4 --out outputs/adapter

Training on device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:02<00:00, 117.50it/s]
Adding EOS to train dataset: 100% 3000/3000 [00:00<00:00, 26967.35 examples/s]
Tokenizing train dataset: 100% 3000/3000 [00:03<00:00, 907.46 examples/s]
Building labels for train dataset: 100% 3000/3000 [00:01<00:00, 2736.20 examples/s]
Truncating train dataset: 100% 3000/3000 [00:00<00:00, 3322.08 examples/s]
Dropping fully masked examples from train dataset: 100% 3000/3000 [00:00<00:00, 4767.65 examples/s]
Adding EOS to eval dataset: 100% 200/200 [00:00<00:00, 19766.74 examples/s]
Tokenizing eval dataset: 100% 200/200 [00:00<00:00, 474.50 examples/s]
Building labels for eval dataset: 100% 200/200 [00:00<00:00, 1388.89 examples/s]
Truncating eval dataset: 100% 200/200 [00:00<00:00, 1795.80 examples/s]
Dropping fully masked examples from eval dataset: 100% 200/200 [00:00<00:00, 2473.61 examples/s]
{'loss': '0.1311', 'grad_norm': '0.1494', 'le

## 7. AFTER numbers: the fine-tuned model

In [ ]:
!python evaluate.py --test data/test_heldout.jsonl --backend hf --model $MODEL --adapter outputs/adapter --out results/tuned_heldout.json
!python evaluate.py --test data/test_in_schema.jsonl --backend hf --model $MODEL --adapter outputs/adapter --out results/tuned_in_schema.json

## 8. Download your adapter and results

In [ ]:
import shutil
shutil.make_archive(f"{WORK}/adapter", "zip", "outputs/adapter")
shutil.make_archive(f"{WORK}/results", "zip", "results")
print("Created adapter.zip and results.zip in", WORK)
try:
    from google.colab import files
    files.download(f"{WORK}/adapter.zip"); files.download(f"{WORK}/results.zip")
except ImportError:
    print("Kaggle: open the Output tab on the right and download the zip files.")

Created adapter.zip and results.zip in /content


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>